# OpenAtlas — N-ATLaS Hosted Endpoint (Colab)

Serves the real **N-ATLaS LLM** (`NCAIR1/N-ATLaS`, Llama-3 8B, 4-bit) plus all **four N-ATLaS ASR models**
(`NCAIR1/Yoruba-ASR`, `NCAIR1/Hausa-ASR`, `NCAIR1/Igbo-ASR`, `NCAIR1/NigerianAccentedEnglish` — all Whisper-small
fine-tunes) behind one FastAPI server, tunneled to a public HTTPS URL.

This is the thing the OpenAtlas SDK's `chat()` and `transcribe()` methods call. It is **not** persistent —
free Colab kills the session on idle timeout or after ~12h, and the tunnel URL changes every run. Treat this
as the "keep it warm for the demo / submission window" deployment, not production hosting (see the
Architecture doc's RunPod Serverless plan for the real persistent version — this notebook is the fast path
to a *working, genuine* N-ATLaS endpoint today).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Request access to `NCAIR1/N-ATLaS` on Hugging Face (one-click "Agree and access repository" — approval is usually instant).
3. Colab secrets (key icon, left sidebar) → add:
   - `HF_TOKEN` — a Hugging Face **read** token, notebook access ON
   - `NATLAS_API_KEY` — any long random string you make up; this becomes the API key OpenAtlas developers send
4. Runtime → Run all.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn[standard] \
    python-multipart nest-asyncio pyngrok soundfile librosa
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!ffmpeg -version | head -1  # audio decoding (webm, m4a, mp3, ...) goes through ffmpeg


## 2. Load secrets

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
NATLAS_API_KEY = userdata.get('NATLAS_API_KEY')
assert HF_TOKEN, "Add HF_TOKEN in Colab secrets (key icon, left sidebar)"
assert NATLAS_API_KEY, "Add NATLAS_API_KEY in Colab secrets — any long random string"

os.environ["HF_TOKEN"] = HF_TOKEN
print("Secrets loaded.")


## 3. Load the N-ATLaS LLM (4-bit, T4)

This is the actual reasoning model — the only thing allowed to answer in `chat()`. No fallback to a
general-purpose model lives in this notebook, on purpose: that's the thing NAIC disqualifies submissions for.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_REPO = "NCAIR1/N-ATLaS"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading N-ATLaS LLM (this takes a few minutes on first run)...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_REPO, token=HF_TOKEN)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_REPO,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
llm_model.eval()
print("N-ATLaS LLM loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 4. Load the four N-ATLaS ASR models (Whisper-small fine-tunes)

In [ ]:
from transformers import pipeline

ASR_REPOS = {
    "yo":    "NCAIR1/Yoruba-ASR",
    "ha":    "NCAIR1/Hausa-ASR",
    "ig":    "NCAIR1/Igbo-ASR",
    "en-ng": "NCAIR1/NigerianAccentedEnglish",
}

asr_pipelines = {}
for lang_code, repo in ASR_REPOS.items():
    print(f"Loading {repo} ({lang_code})...")
    asr_pipelines[lang_code] = pipeline(
        "automatic-speech-recognition",
        model=repo,
        token=HF_TOKEN,
        device=0 if torch.cuda.is_available() else -1,
    )

print("All 4 ASR models loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. FastAPI server

Mirrors the API surface in `OpenAtlas_Architecture_DevPlan.md` section 3, so the SDK's `chat()` and
`transcribe()` methods need zero changes to point at this instead of the final RunPod deployment later —
only `OPENATLAS_BASE_URL` changes.

In [ ]:
import os, subprocess, tempfile, threading, time
from fastapi import FastAPI, Header, HTTPException, UploadFile, File, Form
from pydantic import BaseModel
from typing import List, Optional
import numpy as np

app = FastAPI(title="OpenAtlas N-ATLaS Endpoint")

# One GPU job at a time. Requests run in parallel threads; two generate()/ASR calls at once on a
# 16 GB T4 can run out of memory, so they queue here instead.
gpu_lock = threading.Lock()

SAMPLE_RATE = 16000   # what the Whisper models use
PIECE_SECONDS = 25    # long audio is cut into plain pieces of this length (see deploy/REPORT.md, KI-11)

def decode_audio(raw: bytes) -> np.ndarray:
    """Any format ffmpeg reads (wav, mp3, ogg, flac, webm, m4a) -> 16 kHz mono float32."""
    with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as f:
        f.write(raw)
        path = f.name
    try:
        out = subprocess.run(
            ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "f32le", "pipe:1"],
            capture_output=True, check=True,
        ).stdout
    except subprocess.CalledProcessError as e:
        raise HTTPException(status_code=400, detail=f"Could not decode this audio: {e.stderr.decode(errors='replace')[:300]}")
    finally:
        os.unlink(path)
    return np.frombuffer(out, dtype=np.float32)

def split_audio(samples: np.ndarray) -> list:
    """Plain, non-overlapping pieces of PIECE_SECONDS. Whisper hears 30 s at a time, and the pipeline's own
    chunked mode (chunk_length_s) was measured dropping speech on 38-54 s clips; plain pieces did not.
    A leftover under 2 s is folded into the previous piece rather than sent alone."""
    step = PIECE_SECONDS * SAMPLE_RATE
    pieces = [samples[i:i + step] for i in range(0, len(samples), step)]
    if len(pieces) > 1 and len(pieces[-1]) < 2 * SAMPLE_RATE:
        leftover = pieces.pop()
        pieces[-1] = np.concatenate([pieces[-1], leftover])
    return pieces

def check_key(authorization: Optional[str]):
    if authorization != f"Bearer {NATLAS_API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid or missing API key")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    language: Optional[str] = None
    max_tokens: Optional[int] = 512
    # Sampling settings the OpenAtlas gateway sends; defaults match it (tested on real N-ATLaS weights).
    temperature: Optional[float] = 0.1
    repetition_penalty: Optional[float] = 1.12
    chat_template_kwargs: Optional[dict] = None

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest, authorization: Optional[str] = Header(None)):
    check_key(authorization)
    # N-ATLaS's chat template otherwise hard-codes "Today Date: 26 Jul 2024".
    template_kwargs = {"date_string": time.strftime("%d %b %Y"), **(req.chat_template_kwargs or {})}
    prompt = llm_tokenizer.apply_chat_template(
        [m.dict() for m in req.messages], tokenize=False, add_generation_prompt=True, **template_kwargs
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    t0 = time.time()
    with gpu_lock, torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 512,
            do_sample=(req.temperature or 0) > 0,
            temperature=req.temperature if (req.temperature or 0) > 0 else None,
            repetition_penalty=req.repetition_penalty or 1.0,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    latency = time.time() - t0
    text = llm_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return {
        "content": text.strip(),
        "model": "n-atlas-llm",
        "latency_seconds": round(latency, 2),
    }

@app.post("/v1/audio/transcriptions")
def transcribe(
    audio: UploadFile = File(...),
    language: str = Form(...),
    authorization: Optional[str] = Header(None),
):
    # A plain def: FastAPI runs it in a worker thread, so decoding and the model call don't block the server.
    check_key(authorization)
    if language not in asr_pipelines:
        raise HTTPException(status_code=400, detail=f"language must be one of {list(asr_pipelines)}")
    samples = decode_audio(audio.file.read())
    if samples.size == 0:
        raise HTTPException(status_code=400, detail="The audio is empty.")
    pieces = split_audio(samples)
    t0 = time.time()
    with gpu_lock:
        texts = [asr_pipelines[language]({"array": p, "sampling_rate": SAMPLE_RATE})["text"].strip() for p in pieces]
    latency = time.time() - t0
    return {
        "text": " ".join(t for t in texts if t),
        "language": language,
        "model": f"n-atlas-asr-{language}",
        "audio_seconds": round(samples.size / SAMPLE_RATE, 2),
        "pieces": len(pieces),
        "latency_seconds": round(latency, 2),
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "llm": "NCAIR1/N-ATLaS",
        "asr_languages": list(asr_pipelines.keys()),
    }

print("FastAPI app defined.")


## 6. Smoke test — call it in-process before exposing it publicly

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
headers = {"Authorization": f"Bearer {NATLAS_API_KEY}"}

r = client.post("/v1/chat/completions", headers=headers, json={
    "messages": [{"role": "user", "content": "Ṣe o le ṣàlàyé ìdí tí ọ̀run fi jẹ́ búlúù?"}],
    "max_tokens": 120,
})
print("chat() smoke test:", r.status_code)
print(r.json())

assert r.status_code == 200 and r.json().get("content"), "LLM smoke test failed — do not proceed to tunnel"
print("\nLLM is genuinely responding. Safe to go public.")


## 7. Run the server + public tunnel

Prints `NATLAS_BASE_URL` at the end — that's what goes into the OpenAtlas SDK / starter kits' env var.

In [ ]:
import nest_asyncio, uvicorn, threading, subprocess, time, re

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(5)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
for line in tunnel.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=" * 60)
    print(f"NATLAS_BASE_URL = {public_url}/v1")
    print(f"NATLAS_API_KEY  = {NATLAS_API_KEY}")
    print("=" * 60)
    print("\nPaste these into the OpenAtlas SDK / site env vars.")
    print("This URL dies when this Colab session ends or goes idle — rerun this notebook to get a new one.")
else:
    print("Could not find the tunnel URL in cloudflared output — check the log above.")


## 8. Keep this cell running

Leave this notebook open and this cell running for as long as you need the endpoint up for development,
demo recording, or judges to reach it. Closing the tab / letting the runtime idle out kills it.

In [ ]:
print("Server + tunnel are live. Keep this tab open.")
print("Re-run cell 7 if the tunnel drops (Colab networking hiccup) — same NATLAS_API_KEY, new URL.")
while True:
    time.sleep(60)
